# CropGuard - Baseline Training (Colab T4)

Trains the ResNet50 baseline end to end: data -> split -> train -> ONNX -> INT8 -> holdout eval.

**Runtime > Change runtime type > T4 GPU** before running anything.

| Step | Time (T4) |
|---|---|
| Setup + install | ~3 min |
| Download + extract 54,305 images | ~15 min |
| Validate + split | ~4 min |
| **Train ResNet50, 12 epochs** | **~45-60 min** |
| Export + INT8 + holdout eval | ~10 min |

The dataset is **not** uploaded from your machine - it is pulled from HuggingFace here, and
the split is regenerated from `seed: 42`. Cell 5 verifies by hash that the split is
byte-identical to the local one, so results stay comparable.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU - Runtime > Change runtime type > T4 GPU"

## 2. Clone the repo and install

Colab ships torch with CUDA already, so we install everything *except* torch and then the
package itself with `--no-deps`. Installing `.[train]` normally would drag in a second torch
build and can silently replace the CUDA one with a CPU wheel.

In [ ]:
!git clone -q https://github.com/abhinav7289A/CropGuard.git /content/CropGuard
%cd /content/CropGuard
!git log --oneline -1

!pip install -q pytorch-lightning timm torchmetrics wandb onnx onnxruntime huggingface_hub scikit-learn tqdm pyyaml
!pip install -q -e . --no-deps

import cropguard, torch

print("cropguard", cropguard.__version__, "| torch still CUDA:", torch.cuda.is_available())

## 3. Download the dataset

Pulls `data.zip` (2.2GB) from HuggingFace and extracts the `color` variant into an
ImageFolder tree. Cached, so re-running is cheap.

In [ ]:
import os

os.environ["CROPGUARD_DATA_DIR"] = "/content/cropguard-data"
os.environ["PYTHONIOENCODING"] = "utf-8"

!python -m cropguard.data.download --config configs/base.yaml

## 4. Validate

Integrity, resolution and class-balance gates. Exits non-zero on failure, so if this cell
errors, **stop** - do not train on it.

In [ ]:
!python -m cropguard.data.validate --config configs/base.yaml

## 5. Split - and verify it matches the local split exactly

The leaf-grouped split (see `brain.md` section 3). It is deterministic given `seed: 42`, the
sorted file listing and `leaf-map.json`, so it must reproduce bit-for-bit here. If the hash
differs, something upstream changed and results are **not** comparable to the local run.

In [ ]:
!python -m cropguard.data.split --config configs/base.yaml

import hashlib, json

EXPECTED_SPLIT = "9764d8f2eb2046e9ba91a138e21d472bd6a9e512232431b7d62d252c6ea8efba"
EXPECTED_CLASSES = "f8988b3e341acb2e461ea8dd1caf9d3492a925405a32bcc57a79113237842aa4"

split_sha = hashlib.sha256(open("/content/cropguard-data/splits.json", "rb").read()).hexdigest()
classes_sha = hashlib.sha256(open("configs/classes.json", "rb").read()).hexdigest()

print("splits.json ", split_sha, "MATCH" if split_sha == EXPECTED_SPLIT else "MISMATCH")
print("classes.json", classes_sha, "MATCH" if classes_sha == EXPECTED_CLASSES else "MISMATCH")
print(json.load(open("/content/cropguard-data/split_report.json"))["leakage"])

assert split_sha == EXPECTED_SPLIT, "Split differs from the local run - investigate before training"

## 6. Weights & Biases (optional)

Skip this cell to train without W&B - `train.py` falls back to a local CSV logger.

In [ ]:
import os

# os.environ['WANDB_API_KEY'] = 'paste-your-key'   # or call wandb.login() interactively
# os.environ['WANDB_ENTITY']  = 'your-username'
os.environ.setdefault("WANDB_MODE", "disabled")  # <- delete this line once your key is set
print("W&B mode:", os.environ.get("WANDB_MODE", "online"))

## 7. Persist checkpoints to Google Drive (recommended)

Colab free sessions get reclaimed without warning. Checkpoints are written every epoch and
`--resume` picks up where it stopped - but only if they survive the disconnect.

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/cropguard/checkpoints
    !ln -sfn /content/drive/MyDrive/cropguard/checkpoints /content/CropGuard/checkpoints
    !ls -la /content/CropGuard/checkpoints

## 8. Train the baseline

Colab gives ~2 vCPUs, so we override `num_workers` down from 4 - more workers than cores
adds contention rather than throughput.

**If the session dies:** re-run cells 2-5, then append
`--resume checkpoints/resnet50-baseline/last.ckpt`.

In [ ]:
%%writefile configs/colab_resnet50.yaml
# Colab override: same baseline, dataloader workers matched to the 2-vCPU runtime.
extends: resnet50_baseline.yaml

data:
  num_workers: 2

In [ ]:
# Smoke test first - catches config/data errors in ~30s instead of 45 minutes in.
!python -m cropguard.training.train --config configs/colab_resnet50.yaml --fast-dev-run

In [ ]:
!python -m cropguard.training.train --config configs/colab_resnet50.yaml

## 9. Export to ONNX + INT8

Gated by a PyTorch parity check (max |logit diff| < 1e-3). See `brain.md` section 9 for why
this needs the legacy exporter plus ORT's pre-processing pass.

In [ ]:
import glob

ckpts = sorted(glob.glob("checkpoints/resnet50-baseline/best-*.ckpt"))
assert ckpts, "No checkpoint found - did training finish?"
CKPT = ckpts[-1]
print("using", CKPT)

!python -m cropguard.serving.onnx_export --ckpt "{CKPT}" --out models/cropguard.onnx --quantize
!ls -la models/

## 10. Holdout evaluation - including the INT8 model

`brain.md` section 12 flags that INT8 accuracy on the full holdout had never been measured.
This closes that gap: quantisation is only safe to deploy if the drop is negligible.

In [ ]:
!python -m cropguard.evaluation.predict --model models/cropguard.onnx \
    --split test --out artifacts/preds_fp32.npz --model-version resnet50-fp32

!python -m cropguard.evaluation.predict --model models/cropguard.int8.onnx \
    --split test --out artifacts/preds_int8.npz --model-version resnet50-int8

In [ ]:
import numpy as np
from cropguard.evaluation.predict import load_predictions
from cropguard.evaluation.hypothesis import compare_models

fp32 = load_predictions("artifacts/preds_fp32.npz")
int8 = load_predictions("artifacts/preds_int8.npz")

acc32, acc8 = fp32["correct"].mean(), int8["correct"].mean()
print(f"fp32 holdout accuracy : {acc32:.4f}")
print(f"int8 holdout accuracy : {acc8:.4f}")
print(f"drop                  : {acc32 - acc8:+.4f}")
print()
# Quantisation should NOT be a significant regression. A significant result here is a
# reason not to ship the INT8 model, not a curiosity.
print(compare_models(fp32["correct"], int8["correct"], labels=fp32["labels"]).summary())

In [ ]:
# Per-class report - macro-F1 is the metric that matters under 36x class imbalance.
import json
from sklearn.metrics import f1_score, classification_report

classes = json.load(open("configs/classes.json"))
y_true, y_pred = fp32["labels"], fp32["predictions"]
print("macro-F1 :", f1_score(y_true, y_pred, average="macro"))
print("accuracy :", (y_true == y_pred).mean())
print()
print(classification_report(y_true, y_pred, target_names=classes, digits=3, zero_division=0))

## 11. Save the artifacts

Everything needed downstream: both ONNX graphs and the predictions the A/B comparison will
consume. Checkpoints already live on Drive via the symlink in cell 7.

In [ ]:
!mkdir -p /content/drive/MyDrive/cropguard/artifacts
!cp -v models/cropguard*.onnx /content/drive/MyDrive/cropguard/artifacts/ || true
!cp -v artifacts/preds_*.npz   /content/drive/MyDrive/cropguard/artifacts/ || true
!ls -la /content/drive/MyDrive/cropguard/artifacts/

---
## 12. OPTIONAL - the leakage ablation

**The highest-value extra experiment in this project.** One more training run quantifies how
much the standard PlantVillage split inflates accuracy.

Trains the *identical* model on a naive stratified split, where 74.2% of test images share a
leaf with training. The accuracy gap is the inflation.

Expect the naive split to score **higher** - that is the point. It is measuring memorisation.

**Warning:** this overwrites `splits.json`. Re-run cell 5 afterwards to restore the grouped
split before doing anything else.

In [ ]:
# Rebuild the split WITHOUT leaf grouping, then retrain the same architecture.
!python -m cropguard.data.split --config configs/base.yaml --strategy stratified

import json

print(json.load(open("/content/cropguard-data/split_report.json"))["leakage"])

In [ ]:
%%writefile configs/colab_resnet50_leaky.yaml
# Identical to the baseline in every respect except the split it trains on.
extends: resnet50_baseline.yaml

experiment_name: resnet50-baseline-naive-split

data:
  num_workers: 2

In [ ]:
!python -m cropguard.training.train --config configs/colab_resnet50_leaky.yaml

In [ ]:
# Compare the two holdout accuracies:
#   grouped split -> honest generalisation
#   naive split   -> inflated by leaf memorisation
#
# NOTE: these are DIFFERENT test sets, so this is a descriptive comparison, not a paired
# statistical test. McNemar requires the same holdout and does not apply here.
print("grouped-split holdout accuracy :", acc32)
print("naive-split   holdout accuracy : see test_acc printed by the run above")
print()
print("The gap is the accuracy inflation caused by leaf leakage.")

---
### After this notebook

1. Download the artifacts from Drive.
2. Deploy: point `CROPGUARD_MODEL_PATH` at `cropguard.int8.onnx`, or push the weights to HF
   Hub and set `CROPGUARD_HF_REPO`.
3. Train the ConvNeXt-Tiny challenger (`configs/convnext_tiny.yaml`, ~2-3 hrs) and run
   `cropguard.evaluation.compare` for the first real A/B result.